# Column reference

**Current state** — `direction` is the open Supertrend flip and sets the structure:
**UP → bull put**, **DOWN → bear call**.

| Column | Meaning |
|---|---|
| `symbol` · `direction` | Ticker, and which way the open flip points |
| `trend_age_days` | Calendar days since the flip (weekends count) |
| `spot` · `flip_level` | Latest close, and the Supertrend level that would invalidate the trend |
| `buffer_to_flip_pct` | Distance from spot to that level. **Small = about to invalidate** (NIFTY sat at 0.27% while topping the containment table) |
| `moved_since_entry_pct` | Signed *for* the position: positive = trend working, negative = underwater |
| `sector` · `beta` | From a hand-maintained map; beta vs NIFTY over ~250 days. Beta makes the 50/50 split *shock*-balanced, not just count-balanced |
| `realized_vol_pct` · `realized_vol_rank` | Annualized `window`-day realized vol, and where TODAY's value ranks (0-100) against this stock's own trailing ~2yr history. **Backfillable interim proxy for IV rank — not IV itself, no established threshold, informational only for now** (see Stage 8f) |

**Duration context** — descriptive only.

| Column | Meaning |
|---|---|
| `median_duration` | This stock's typical flip length |
| `past_median` · `past_75th_pct` · `past_90th_pct` | Has the open trend outlived that checkpoint? |
| `runway_left_vs_median` · `_p75` · `_p90` | `checkpoint − age` in signed days; negative = past it |

> `reversion_test.py` and `exhaustion_test.py` both tested whether crossing these
> predicts a reversal. **It does not.** `past_median = True` is not a warning and a
> negative runway is not an exit signal.

**Does the signal work here?** — all on closed flips in the *currently open* direction.

| Column | Meaning |
|---|---|
| `dir_flips` | Sample size; under 12 the edge columns return `None` |
| `dir_win_rate` | % of those flips that moved as the signal said |
| `dir_median_move` | Median favourable move. **Standalone for this side — not a comparison to the opposite side.** Negative = the typical flip goes the wrong way |

**Strike selection** — containment is the primary metric. A credit spread is a
"price won't breach my strike" bet, not a directional one.

| Column | Meaning |
|---|---|
| `contained_5pct` · `contained_6pct` | % of same-direction flips whose worst adverse excursion stayed inside that buffer |
| `worst_adverse` | Largest adverse excursion in 4yr — watch for fat tails behind good containment |
| `worst_gap` | Largest overnight gap against the position (open vs prev close) |
| `worst_day` | Largest full close-to-close day against it. Exists because `worst_gap` misses intraday collapses — PIIND's 8.97% fall was only a 1.28% gap |
| `pct_within_30d` | % resolved inside 30 days — matters for monthly expiries |
| `short_strike` · `strike_buffer_pct` | Strike implied by this stock's own containment, off live spot. Says nothing about whether the premium there is worth taking (needs IV — not built) |

**Was this flip normal?**

| Column | Meaning |
|---|---|
| `entry_day_move` | Signed move on the day the open flip was born |
| `born_on_outlier_day` | `abs(entry_day_move) > 4%`. The containment stats above come from *ordinary* flips and do not describe a trend that began mid-crash |

**Wrong half of a good stock?**

| Column | Meaning |
|---|---|
| `opp_win_rate` · `opp_median_move` | The direction you are **not** trading. **Lower is not better** — these say nothing about the current trade. Their only job: current weak + opposite strong → watchlist; weak both ways → no edge |

**Verdicts**

| Column | Meaning |
|---|---|
| `edge_flag` | Empty = tradeable · `⚠ weak side` = watchlist · `⚠ NO EDGE EITHER SIDE` = fails both ways · `⚠ flip born on an outlier day` appended to any of these · `insufficient history` |
| `setup_score` | 0–100 composite (containment ×3, gap ×2, day ×2, win ×2, move ×1). **Relative to today's screen only** — not comparable across days |
| `reason` | Plain-language why, built from this stock's own numbers |

Bar for "has edge": `dir_win_rate >= 50` **and** `dir_median_move > 0`. Deliberately
plain, not optimised — nothing here is walk-forward validated. Names on the boundary
(BOSCHLTD `50.0/+0.08` passes, KOTAKBANK `50.0/−0.10` fails) are indistinguishable.

---

**Caveats**

1. **In-sample and descriptive** — same ~4 years the ranking is built from, no
   walk-forward validation. What the signal *has* done, not what it will do.
2. **Duration columns are pooled; edge columns are split by direction.**
3. **Containment spans the whole flip, not 30 days** — conservative.
4. **Not here at all:** earnings dates, real IV rank (a forward IV-history recorder started — `scripts/record_iv_snapshot.py` — but needs months to accumulate a usable percentile; `realized_vol_rank` above is the interim proxy), options liquidity, exits.


In [1]:
# It reads the saved credit-spread ranking, drops anything flagged for gap risk, 
# takes the top 50 by score, 
# then force-adds NIFTY/BANKNIFTY (same gap-risk filter applied to them too), 
# and merges into one deduplicated final_candidates list. No API calls here — pure filtering on the 
# last saved ranking file.


# autoreload: re-imports any changed src/*.py automatically, so an old kernel
# can never run stale module code. Without this, editing src/ and re-running
# without a kernel restart silently uses the OLD version — which is how 31
# reports once got rewritten missing their new columns.
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")

import pandas as pd
from src.daily_screener import screen_list

credit_spread_ranking = pd.read_excel("../exports/RANKING_credit_spreads.xlsx")

# Keep only gap-risk-clean stocks, take the top 50
clean_candidates = credit_spread_ranking[
    credit_spread_ranking["gap_risk_flag"].isna() | (credit_spread_ranking["gap_risk_flag"] == "")
]
top_50 = clean_candidates.head(50)

# Always include NIFTY and BANKNIFTY, gap-risk-checked the same way
must_include = credit_spread_ranking[credit_spread_ranking["symbol"].isin(["NIFTY", "BANKNIFTY"])]
must_include_clean = must_include[must_include["gap_risk_flag"].isna() | (must_include["gap_risk_flag"] == "")]

final_candidates = pd.concat([top_50, must_include_clean]).drop_duplicates(subset="symbol").reset_index(drop=True)

print("SORTED BY: credit_spread_score, descending (inherited from the ranking file).")
print("           NIFTY/BANKNIFTY appended at the end if not already in the top 50.")
final_candidates

SORTED BY: credit_spread_score, descending (inherited from the ranking file).
           NIFTY/BANKNIFTY appended at the end if not already in the top 50.


,symbol,credit_spread_score,gap_risk_flag,day_risk_flag,Total flips (closed),Median duration (days),% completed within a monthly cycle,Median max_adverse_pct,Worst max_adverse_pct (4yr),Worst single gap (4yr),Worst single-day move against (4yr),current_trend_age_days,trend_age_note
0,NESTLEIND,86.3,NaN,NaN,57,24.2,57.9,2.74,7.61,4.37,6.37,12,NaN
1,ITC,85.4,NaN,NaN,57,22.0,63.2,2.55,9.56,4.68,4.89,19,NaN
2,SUNPHARMA,85.1,NaN,NaN,53,20.0,66.0,2.57,7.31,3.53,4.51,85,"note: aged trend — tested data shows momentum,..."
3,BANKNIFTY,82.5,NaN,NaN,73,20.2,69.9,1.53,7.15,3.73,3.96,4,NaN
4,COLPAL,81.9,NaN,NaN,63,19.2,66.7,3.12,6.25,2.32,4.78,6,NaN
5,DRREDDY,81.4,NaN,NaN,62,22.2,69.4,2.84,7.88,4.27,4.88,15,NaN
6,HAVELLS,81.3,NaN,NaN,65,22.0,67.7,3.28,6.58,4.42,4.54,6,NaN
7,SHREECEM,80.6,NaN,NaN,62,22.4,69.4,3.12,7.36,3.82,5.63,4,NaN
8,ULTRACEMCO,78.8,NaN,NaN,70,19.0,65.7,2.43,7.21,3.61,6.86,11,NaN
9,DABUR,77.3,NaN,NaN,67,19.2,71.6,2.41,7.34,2.66,4.35,26,NaN


In [2]:
# For each of the ~50 symbols it looks up the right instrument 
# token (handling NIFTY/BANKNIFTY specially since indices live in a different token space), 
# fetches only the recent catch-up window of 4H candles from Kite (not the full 2022-to-now
# history - process_stock_incremental() merges this onto existing saved data instead of
# replacing it, same fix already applied to the debit spread/option buying/pledged pipelines,
# here for efficiency rather than a data-loss bug),
# reruns Supertrend → flip detection → monthly range → summary stats on the FULL merged
# history, and updates that symbol's _full_report.xlsx. This is the only block that talks to the API

from src.auth import get_kite
from src.instruments import build_nse_token_lookup, get_index_token
from src.scale_pipeline import process_stock_incremental
from datetime import datetime

kite = get_kite()
token_lookup = build_nse_token_lookup(kite)

symbols_to_refresh = final_candidates["symbol"].tolist()
index_symbols = {"NIFTY": ("NIFTY 50", "NSE"), "BANKNIFTY": ("NIFTY BANK", "NSE")}

for i, symbol in enumerate(symbols_to_refresh, start=1):
    try:
        if symbol in index_symbols:
            tradingsymbol, exchange = index_symbols[symbol]
            token = get_index_token(kite, tradingsymbol, exchange)
        else:
            token = token_lookup.get(symbol)

        if token is None:
            print(f"[{i}/{len(symbols_to_refresh)}] {symbol} — no token found, skipping")
            continue

        process_stock_incremental(symbol, token, kite, to_date=datetime.now())
        print(f"[{i}/{len(symbols_to_refresh)}] {symbol} — refreshed")
    except Exception as e:
        print(f"[{i}/{len(symbols_to_refresh)}] {symbol} — FAILED: {e}")

Saved token expired, need fresh login.
Login here: https://kite.zerodha.com/connect/login?api_key=bz1cszn3hpbgzkva&v=3
Fetching 2026-06-29 to 2026-10-28...
Saved 2377 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/NESTLEIND_4hour.csv
Saved full report to /Users/hgmac/Documents/3_trading/Quant/exports/NESTLEIND_full_report.xlsx
[1/50] NESTLEIND — refreshed
Fetching 2026-06-29 to 2026-10-28...
Saved 2377 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/ITC_4hour.csv
Saved full report to /Users/hgmac/Documents/3_trading/Quant/exports/ITC_full_report.xlsx
[2/50] ITC — refreshed
Fetching 2026-06-29 to 2026-10-28...
Saved 2377 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/SUNPHARMA_4hour.csv
Saved full report to /Users/hgmac/Documents/3_trading/Quant/exports/SUNPHARMA_full_report.xlsx
[3/50] SUNPHARMA — refreshed
Fetching 2026-06-29 to 2026-10-28...
Saved 2377 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/BANKNIFTY_4hour.csv
Saved full report t

In [3]:
# reads the just-refreshed reports and reports current state:
# for each symbol, current trend direction, how many days the open trend has run,
# whether that age has crossed the stock's own median/75th/90th percentile duration,
# and runway_left_vs_median/p75/p90 - the same comparison as a signed number of
# days instead of three separate booleans (negative = already past that checkpoint) -
# sorted so the youngest (most recently flipped) trends show first. Same caveat as
# before: crossing a checkpoint is NOT a reversal/exit signal (tested, see README) -
# read runway as "how this compares to history," not as a warning.
#
# NEW - directional edge. The credit spread ranking read in cell 1 is built purely
# from risk-containment metrics (median/worst adverse, worst gap, duration, % within
# a monthly cycle) POOLED across both directions, and contains no profitability term
# at all. So a stock whose UP flips work and whose DOWN flips don't scores exactly
# like one where both work, and the shortlist gives no way to tell them apart. These
# columns answer, for the side that is actually open right now:
#   dir_win_rate/dir_median_move  - did the underlying actually move the way the
#       signal said? (directional read)
#   contained_5pct/contained_6pct - did price stay inside that buffer? This is what a
#       credit spread actually needs, since the structure is a "won't breach my
#       strike" bet, not a directional one. Read containment as PRIMARY and win rate
#       as a secondary tilt.
#   opp_win_rate/opp_median_move  - the other side, for contrast
# Caveat: in-sample and descriptive, same family as the runway columns - computed on
# the same ~4 years the ranking is built from, no walk-forward validation. It says
# what the signal HAS done here, not what it WILL do.

from src.daily_screener import screen_list

status = screen_list(symbols_to_refresh)

print("SORTED BY: trend_age_days, ascending — most recently flipped first.")
status.sort_values("trend_age_days")


SORTED BY: trend_age_days, ascending — most recently flipped first.


,symbol,direction,trend_age_days,median_duration,past_median,past_75th_pct,past_90th_pct,runway_left_vs_median,runway_left_vs_p75,runway_left_vs_p90,...,contained_6pct,worst_adverse,worst_gap,worst_day,pct_within_30d,entry_day_move,born_on_outlier_day,opp_win_rate,opp_median_move,edge_flag
23,APOLLOHOSP,UP,0,19.5,False,False,False,19.5,34.4,51.0,...,100.0,5.95,4.34,5.19,61.8,2.38,False,41.2,-0.83,
47,FORTIS,UP,0,20.8,False,False,False,20.8,50.6,75.7,...,88.5,8.16,3.84,6.21,57.7,3.09,False,38.5,-2.56,
46,LT,UP,0,18.6,False,False,False,18.6,27.8,52.0,...,97.1,6.58,3.40,3.43,71.4,3.32,False,45.7,-0.45,
43,TCS,DOWN,0,14.2,False,False,False,14.2,29.0,48.2,...,97.5,6.86,2.71,4.49,75.0,-3.56,False,53.7,0.26,
24,CIPLA,UP,1,19.9,False,False,False,18.9,30.7,44.7,...,100.0,4.84,3.54,3.98,73.5,-0.36,False,44.1,-0.70,⚠ NO EDGE EITHER SIDE — signal has not worked ...
7,SHREECEM,UP,4,22.4,False,False,False,18.4,36.5,50.2,...,90.3,7.36,3.82,4.52,67.7,1.97,False,61.3,1.90,
17,KOTAKBANK,DOWN,4,18.2,False,False,False,14.2,28.1,40.8,...,86.5,7.52,3.55,4.81,62.2,-1.84,False,52.6,0.61,⚠ weak side — opposite direction is the strong...
37,NIFTY,DOWN,5,15.2,False,False,False,10.2,19.2,42.4,...,100.0,5.95,2.32,3.16,79.5,-0.39,False,67.5,0.57,
35,ICICIGI,DOWN,5,20.0,False,False,False,15.0,26.2,54.9,...,96.9,6.56,3.32,7.39,68.8,-0.61,False,39.4,-1.54,
36,SENSEX,DOWN,5,16.0,False,False,False,11.0,20.9,42.4,...,100.0,5.55,2.31,3.24,79.5,-0.17,False,67.5,0.68,


In [4]:
# Splits the screen into the three buckets that call for DIFFERENT action, so a
# no-edge name can't sit next to a good one looking identical (which is exactly what
# the pooled ranking lets happen):
#
#   tradeable - the open direction has historically worked on this stock
#   watchlist - wrong side of a stock whose OTHER side works. Not a bad stock, just
#               the wrong half of it right now - revisit when it flips, don't discard
#   no_edge   - the signal has not worked in EITHER direction. This is the case the
#               ranking is blindest to: score only measures how well-behaved price
#               is, so a stock with tight, gap-free moves and a signal that's wrong
#               both ways scores well and looks like a good candidate. Waiting for a
#               flip does not fix these - the signal itself doesn't work here.
#
# Nothing is deleted - no_edge is returned, not dropped, so the exclusion stays
# visible and auditable. Same reasoning as the scorer's gap_risk_flag, which warns
# without removing the row.
#
# See the column reference at the top of this notebook for what each field means.

from src.daily_screener import split_by_edge

tradeable, watchlist, no_edge = split_by_edge(status)

edge_cols = ["symbol", "direction", "trend_age_days", "dir_flips", "dir_win_rate",
             "dir_median_move", "contained_5pct", "contained_6pct", "worst_adverse",
             "worst_gap", "worst_day", "pct_within_30d", "entry_day_move",
             "born_on_outlier_day", "opp_win_rate", "opp_median_move"]

print(f"=== NO EDGE EITHER SIDE ({len(no_edge)}) — do not trade, flipping won't help ===")
print("    SORTED BY: nothing — falls through in screen order (i.e. ranking order)")
display(no_edge[edge_cols])

print(f"\n=== WATCHLIST ({len(watchlist)}) — wrong side now, other side works ===")
print("    SORTED BY: opp_win_rate, descending — best flip candidates first")
display(watchlist[edge_cols].sort_values("opp_win_rate", ascending=False))

print(f"\n=== TRADEABLE ({len(tradeable)}) — open direction has worked historically ===")
print("    SORTED BY: contained_6pct desc, then dir_win_rate desc — containment is the")
print("               primary metric, so a top win rate can still sit low in this table")
display(tradeable[edge_cols].sort_values(["contained_6pct", "dir_win_rate"], ascending=False))

# Crash-born flips, called out separately. These sit INSIDE the buckets above (a
# stock with real edge still lands in tradeable) but the containment stats there
# are drawn from ordinary flips and do not describe a trend that began mid-crash -
# PIIND topped this shortlist on 2026-08-12 on a flip created that morning by a
# -8.97% fall. Printed last so it's the thing left on screen before you act.
outliers = status[status["born_on_outlier_day"] == True]
if len(outliers):
    print(f"\n=== ⚠ FLIPS BORN ON AN OUTLIER DAY ({len(outliers)}) — stats above do not describe these ===")
    print("    SORTED BY: entry_day_move, ascending — biggest fall first")
    display(outliers[["symbol", "direction", "trend_age_days", "entry_day_move",
                      "dir_win_rate", "contained_6pct", "worst_gap", "worst_day"]]
            .sort_values("entry_day_move"))


=== NO EDGE EITHER SIDE (4) — do not trade, flipping won't help ===
    SORTED BY: nothing — falls through in screen order (i.e. ranking order)


,symbol,direction,trend_age_days,dir_flips,dir_win_rate,dir_median_move,contained_5pct,contained_6pct,worst_adverse,worst_gap,worst_day,pct_within_30d,entry_day_move,born_on_outlier_day,opp_win_rate,opp_median_move
16,GRASIM,DOWN,43,33,30.3,-1.31,97.0,100.0,5.05,3.44,5.27,78.8,-3.27,False,47.1,-0.25
24,CIPLA,UP,1,34,50.0,-0.08,100.0,100.0,4.84,3.54,3.98,73.5,-0.36,False,44.1,-0.70
33,ALKEM,DOWN,29,31,48.4,-1.47,74.2,87.1,13.46,3.33,6.60,67.7,-2.09,False,37.5,-0.50
34,HINDUNILVR,DOWN,6,35,45.7,-0.10,91.4,94.3,8.27,4.18,4.93,65.7,-1.49,False,41.7,-0.22



=== WATCHLIST (15) — wrong side now, other side works ===
    SORTED BY: opp_win_rate, descending — best flip candidates first


,symbol,direction,trend_age_days,dir_flips,dir_win_rate,dir_median_move,contained_5pct,contained_6pct,worst_adverse,worst_gap,worst_day,pct_within_30d,entry_day_move,born_on_outlier_day,opp_win_rate,opp_median_move
32,AXISBANK,DOWN,7,37,43.2,-1.28,91.9,94.6,9.03,4.56,5.43,75.7,-1.88,False,65.8,1.98
42,JINDALSTEL,DOWN,27,32,37.5,-0.83,59.4,78.1,10.29,3.88,7.70,75.0,-2.26,False,63.6,3.13
10,BAJAJFINSV,DOWN,82,29,44.8,-0.77,82.8,89.7,7.86,3.53,4.91,58.6,-3.86,False,63.3,1.14
21,OBEROIRLTY,DOWN,27,31,48.4,-0.20,54.8,71.0,12.92,4.81,6.11,67.7,-1.77,False,62.5,2.83
0,NESTLEIND,DOWN,12,28,46.4,-0.05,89.3,100.0,5.96,4.15,3.61,71.4,-6.37,True,62.1,1.88
41,CONCOR,DOWN,61,33,42.4,-0.60,81.8,84.8,22.36,3.29,6.35,69.7,-1.60,False,61.8,1.25
40,EICHERMOT,DOWN,6,34,47.1,-0.53,76.5,88.2,7.42,3.49,3.71,82.4,-0.71,False,60.0,1.33
2,SUNPHARMA,DOWN,85,26,42.3,-1.24,88.5,92.3,7.31,3.53,4.51,69.2,-2.11,False,59.3,0.26
45,FINNIFTY,DOWN,7,41,48.8,-0.01,97.6,97.6,7.04,2.35,3.42,82.9,-1.18,False,57.1,0.46
4,COLPAL,DOWN,7,31,41.9,-0.99,90.3,100.0,5.51,2.12,3.59,64.5,-0.82,False,56.2,0.34



=== TRADEABLE (31) — open direction has worked historically ===
    SORTED BY: contained_6pct desc, then dir_win_rate desc — containment is the
               primary metric, so a top win rate can still sit low in this table


,symbol,direction,trend_age_days,dir_flips,dir_win_rate,dir_median_move,contained_5pct,contained_6pct,worst_adverse,worst_gap,worst_day,pct_within_30d,entry_day_move,born_on_outlier_day,opp_win_rate,opp_median_move
28,TITAN,DOWN,55,32,59.4,0.79,96.9,100.0,5.65,2.94,5.75,81.2,-0.79,False,60.6,1.26
29,JSWSTEEL,UP,12,34,58.8,0.61,91.2,100.0,5.90,3.31,4.39,58.8,1.39,False,44.1,-0.11
36,SENSEX,DOWN,5,39,56.4,0.66,97.4,100.0,5.55,2.31,3.24,79.5,-0.17,False,67.5,0.68
37,NIFTY,DOWN,5,39,56.4,0.27,97.4,100.0,5.95,2.32,3.16,79.5,-0.39,False,67.5,0.57
19,ICICIBANK,UP,22,36,55.6,0.48,97.2,100.0,5.22,2.95,3.13,72.2,0.40,False,44.4,-1.01
23,APOLLOHOSP,UP,0,34,52.9,0.28,79.4,100.0,5.95,4.34,5.19,61.8,2.38,False,41.2,-0.83
49,NIFTYNXT50,DOWN,30,42,52.4,0.21,97.6,97.6,9.24,2.80,3.88,88.1,-2.02,False,72.1,1.08
43,TCS,DOWN,0,40,62.5,1.64,95.0,97.5,6.86,2.71,4.49,75.0,-3.56,False,53.7,0.26
3,BANKNIFTY,DOWN,5,36,52.8,0.87,97.2,97.2,7.15,3.73,3.96,72.2,-1.16,False,64.9,1.16
46,LT,UP,0,35,60.0,1.68,94.3,97.1,6.58,3.40,3.43,71.4,3.32,False,45.7,-0.45



=== ⚠ FLIPS BORN ON AN OUTLIER DAY (3) — stats above do not describe these ===
    SORTED BY: entry_day_move, ascending — biggest fall first


,symbol,direction,trend_age_days,entry_day_move,dir_win_rate,contained_6pct,worst_gap,worst_day
0,NESTLEIND,DOWN,12,-6.37,46.4,100.0,4.15,3.61
48,BAJAJHLDNG,UP,13,4.22,52.9,85.3,4.84,8.41
22,HCLTECH,UP,15,8.57,58.8,97.1,4.26,5.36


In [5]:
# ============================================================================
#  THE FINAL LIST — this is the only cell you need to read.
# ============================================================================
# Everything above is the working-out. This applies the whole filter chain and
# returns the book to trade, balanced 50/50 between bull puts and bear calls.
#
# Filters applied, in order (each one is reported below, nothing is silent):
#   1. no edge either side   - signal has never worked on this stock
#   2. wrong side            - open direction is this stock's weak half
#   3. crash-born flip       - flip created by an outlier day; the containment
#                              stats come from ordinary flips and don't apply
#   4. thin buffer           - spot too close to the level that invalidates it
#   5. trend underwater      - thesis already failing before entry
#   6. sector cap            - per DIRECTION (opposite bets on one sector hedge
#                              each other, they don't concentrate risk)
#   7. 50/50 direction       - so one market-wide shock can't hit everything
#
# Identical to `python scripts/daily_list.py`, which is the schedulable version.

import importlib
import src.daily_screener, src.market_context, src.entry_advisor
importlib.reload(src.daily_screener)
importlib.reload(src.market_context)
importlib.reload(src.entry_advisor)
from src.entry_advisor import build_entry_list

final, excluded, summary = build_entry_list(symbols_to_refresh, n_total=12, max_per_sector=2)

print(f"{summary['up']} BULL PUT  /  {summary['down']} BEAR CALL"
      f"      mean beta {summary['mean_beta_up']} UP vs {summary['mean_beta_down']} DOWN")
if summary["up"] != summary["down"]:
    print("  ** NOT BALANCED — too few DOWN-side candidates survived the filters.")
    print("     Trade fewer positions rather than forcing in a weak bear call.")
print("SORTED BY: direction (DOWN before UP, alphabetical — not a quality ordering),")
print("           then setup_score descending WITHIN each direction block.")
print("           Which names got PICKED was decided by setup_score desc overall.")
print()

display(final[["symbol", "direction", "trend_age_days", "sector", "short_strike", "spot",
               "buffer_to_flip_pct", "dir_win_rate", "dir_median_move",
               "contained_6pct", "worst_gap", "worst_day", "beta",
               "realized_vol_rank", "setup_score"]])

# The WHY for each, so the list can be acted on without re-deriving anything.
for _, r in final.iterrows():
    print(f"\n{r['symbol']} — short strike ~{r['short_strike']:,.1f} "
          f"({r['strike_buffer_pct']}% from spot)")
    print(f"   {r['reason']}")

print("\n" + "=" * 78)
print("EXCLUDED — considered and rejected")
for key, names in excluded.items():
    if names:
        print(f"  {key:24s} {', '.join(names)}")

print("\nSTILL MANUAL, not checked by any of the above:")
print("  earnings / corporate actions · IV rank · options liquidity · exits")
print("All statistics are in-sample and descriptive — what the signal HAS done,")
print("not what it will do.")


6 BULL PUT  /  6 BEAR CALL      mean beta 0.97 UP vs 0.98 DOWN
SORTED BY: direction (DOWN before UP, alphabetical — not a quality ordering),
           then setup_score descending WITHIN each direction block.
           Which names got PICKED was decided by setup_score desc overall.



,symbol,direction,trend_age_days,sector,short_strike,spot,buffer_to_flip_pct,dir_win_rate,dir_median_move,contained_6pct,worst_gap,worst_day,beta,realized_vol_rank,setup_score
37,NIFTY,DOWN,5,INDEX,23985.0,22627.35,2.32,56.4,0.27,100.0,2.32,3.16,1.00,76.2,76.9
43,TCS,DOWN,0,IT,2555.1,2410.50,7.18,62.5,1.64,97.5,2.71,4.49,0.80,95.8,75.8
15,HDFCBANK,DOWN,7,BANK,722.9,681.95,3.05,54.5,0.15,97.0,2.11,3.58,1.20,69.0,64.0
9,DABUR,DOWN,27,FMCG,380.3,358.80,3.88,54.5,0.83,97.0,2.66,3.66,0.91,73.0,63.7
28,TITAN,DOWN,55,CONSUMER,4582.4,4323.00,2.39,59.4,0.79,100.0,2.94,5.75,1.01,71.8,62.3
6,HAVELLS,DOWN,7,ELECTRICALS,1075.9,1015.00,5.71,56.2,1.54,96.9,2.59,4.54,0.99,93.6,61.9
18,BHARTIARTL,UP,22,TELECOM,1826.6,1943.20,3.58,59.4,1.27,96.9,2.44,3.39,0.67,91.0,71.5
19,ICICIBANK,UP,22,BANK,1293.3,1375.90,2.16,55.6,0.48,100.0,2.95,3.13,0.91,67.8,70.8
46,LT,UP,0,INFRA,3578.8,3807.20,4.41,60.0,1.68,97.1,3.40,3.43,1.42,57.4,68.8
29,JSWSTEEL,UP,12,METALS,1166.5,1241.00,5.26,58.8,0.61,100.0,3.31,4.39,1.16,87.8,65.8



NIFTY — short strike ~23,985.0 (6.0% from spot)
   BEAR CALL — trend is DOWN, sell calls above. Flip is 5d old vs a 15.2d median. DOWN flips here win 56.4% with a +0.27% median move (39 samples). 100.0% of them stayed inside 6% (worst 5.95%). Worst gap 2.32%, worst single day 3.16%. 79.5% resolved within 30d. Spot 22627.35 sits 2.32% from the 23152.33 flip level. Trend is slightly underwater (-1.00% since flip). Beta 1.0. Realized vol 14.7% (annualized), elevated vs its own 2yr history (76.2th pct — proxy only, not IV, no established threshold yet).

TCS — short strike ~2,555.1 (6.0% from spot)
   BEAR CALL — trend is DOWN, sell calls above. Flip is 0d old vs a 14.2d median. DOWN flips here win 62.5% with a +1.64% median move (40 samples). 97.5% of them stayed inside 6% (worst 6.86%). Worst gap 2.71%, worst single day 4.49%. 75.0% resolved within 30d. Spot 2410.5 sits 7.18% from the 2583.59 flip level. Trend is slightly underwater (-0.00% since flip). Beta 0.8. Realized vol 38.2% (ann

## Paper journal — every name on the final list, tracked to SL or expiry

Each symbol on the final list above is logged as a trade taken now (one open
trade per symbol — re-appearing tomorrow is the same trade). Every later run
checks all open trades, **whether or not they are still on the list**, for
exactly two exits:

- **SL_HIT** — a completed 4H candle closed beyond the trailing Supertrend level (the flip)
- **EXPIRED** — reached the current month's monthly expiry without an SL hit

`strike_touched` / `expired_beyond_strike` record whether the short strike was
reached — information, not an exit. Forward-only and append-only: a trade
exists because the list showed it that day, so it cannot be rebuilt later
(`logs/credit_spread_journal.csv`, tracked in git). See `src/credit_spread_journal.py`.

In [ ]:
from src.credit_spread_journal import run_journal, summarize

journal, logged, problems = run_journal(kite, final, already_refreshed=symbols_to_refresh)
print(f"New trades logged this run: {', '.join(logged) if logged else 'none'}")
for p in problems:
    print(f"  ** {p}")

open_trades = journal[journal["status"] == "OPEN"]
print(f"\nOPEN — {len(open_trades)} trade(s), most recent first:")
display(open_trades.sort_values("logged_at", ascending=False)[
    ["symbol", "structure", "logged_at", "entry_spot", "short_strike", "initial_sl",
     "expiry", "max_adverse_pct", "strike_touched"]].reset_index(drop=True))

closed = journal[journal["status"] == "CLOSED"]
print(f"\nCLOSED — {len(closed)} trade(s):")
if not closed.empty:
    display(summarize(journal))
    display(closed.sort_values("exit_date", ascending=False)[
        ["symbol", "structure", "logged_at", "exit_date", "exit_reason", "entry_spot",
         "exit_spot", "sl_at_exit", "pct_move", "max_adverse_pct", "strike_touched",
         "expired_beyond_strike", "days_held"]].reset_index(drop=True))